# Real Time pH Control using the Daisy Lab Automation Platform

In this application of the Daisy instrument platform you will utilize the Daisy python library to automatically control the pH of your solution in real-time. You will need a **Daisy Process** instrument with a Hamilton pH meter and 2 **Daisy Peri Ps**.

This system continuously measures pH and responds to deviations by dispensing small doses of acid or base through the respective Daisy P pumps. A PID controller calculates the appropriate dose size based on the magnitude and history of the pH error, while a hysteresis mechanism prevents over-dosing by disarming the controller once the pH returns within an acceptable range and only re-arming it when the error grows large enough to warrant intervention.

Measurements, dosing events, and timestamps are logged to a text file and pH is plotted in real time so you can monitor the system visually throughout the run.

## Import
The Daisy Python Library is already loaded into the Daisy Controller Software, all you have to do is import any external libraries you need. Here we are importing additional libraries to build and update the real-time pH plot, generate timestamps and record time elapsed, and a fast sliding window list used to plot data and calculate PID.

In [ ]:
# Packages to install
%matplotlib widget
import time

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from datetime import datetime
from collections import deque


# Set your output file path and output file name
file_path = "/Users/yourusername/Documents/"
file_name = "pH_output.txt"

## Print instruments found
Print the instruments found to easily verify the index of the Daisy you will be using.

In [ ]:
print("Instruments found:")
for position, (code, inst) in enumerate(zip(self.my_interface.instruments_list[0], self.my_interface.groups[0].inst)):
    print(f"  [{position}]  {code} — {inst.name}")

## Assign your instruments

The order of your instruments and their labels are important to add acid and base when each is called, respectively.

`self.my_interface.groups[0].inst[n]` accesses instruments by their 0-based position in the group.

In [ ]:
# Define your instruments
pH_meter = self.my_interface.groups[0].inst[0]     # Daisy Process
acid_pump = self.my_interface.groups[0].inst[1]    # Daisy Peri P, acid
base_pump = self.my_interface.groups[0].inst[2]    # Daisy Peri P, base

## Calibrate your Peri P pumps

If you need to calculate the calibration value for your solution and tubing reference our **Daisy Peristaltic Pump** quick start guide.

`calibrate(speed_at_60rpm)` is how many ml/min your liquid and tubing delivers at 60 rpm. Here we are using 5.84 ml/min.

In [ ]:
# Peri P
# Set the calibration (ml/min @ 60 rpm) for each Daisy Peri P
acid_pump.calibrate(5.84)
base_pump.calibrate(5.84)

print(f"Acid pump calibration set to {acid_pump.calibration_speed} ml/min @ 60 RPM")
print(f"Base pump calibration set to {base_pump.calibration_speed} ml/min @ 60 RPM")

## Prime both of your Peri P pumps

If you skip priming, dosing events will not result in a change in pH until the tubing line is filled. The parameters for running the peristaltic pumps here will depend on the length of the tubing that needs to be primed. A slow flow rate is recommended to not waste your input solution and the volume can be increased or decreased depending on the length of tubing used.

### Prime acid tubing

Run your acid pump to prime the acid tubing.

In [ ]:
# Parameters for the Daisy Peri Ps are flow rate (ml/min), total volume (ml), and wait
# Wait allows for the pump to finish running before running the next line

acid_pump.run(6.0, 1.5, True)

### Prime base tubing

Run your base pump to prime the base tubing.

In [ ]:
base_pump.run(6.0, 1.5, True)

## Set the parameters for your real time pH control

The Daisy Process has 4 available sensor channels. The channel must be specified otherwise all values will read 99.99.

In [ ]:
# pH meter

# Set the channel your pH meter is connected to
# Default is channel 1, if using 2-4 specify channel
pH_channel = 2

# Take a test measurement of your pH meter
pH, mV, T = pH_meter.measure_parameters(pH_channel)

print(f"pH: {pH:.3f} mV: {mV:.3f} Temperature: {T:.2f} °C")

## Configure control parameters

These parameters control how aggressively the system doses and how tightly it holds the target pH.

**Set Target pH:**
- `target_pH` — the pH you want to maintain
- `epsilon` — error, inner band; dosing deactivates once |error| ≤ epsilon
- `outer_band` — outer band; dosing reactivates once |error| ≥ outer_band (must be ≥ epsilon)

**PID gains:**
- `kp` — proportional gain (how hard to correct a current error)
- `ki` — integral gain (how hard to correct a persistent error over time)
- `kd` — derivative gain (how hard to damp rapid changes)

**Dosing limits:**
- `DROP_ML` — volume of one drop (~0.033 ml)
- `min_ml` / `max_ml` — minimum and maximum dose per dispensing event
- `req_rate_ml_min` — pump speed during dosing
- `cooldown_s` — minimum seconds between doses
- `scale_ml_per_unit` — converts PID output to a dose volume

In [ ]:
# Set your desired pH
target_pH = 7.0

# Set your inner band, epsilon, the margin for error
epsilon = 0.1

# Set your outer band, dosing re-arms outside this band, must be ≥ epsilon
outer_band = 0.5


# Set your PID gains
kp = 0.8
ki = 0.0
kd = 0.0

# Set your dosing parameters
DROP_ML           = 0.033              # ~1 drop in ml
PUMP_MAX_ML_MIN   = 20.0               # pump speed cap (ml/min)
scale_ml_per_unit = 0.10               # dose volume = |PID output| * scale
max_ml            = 10.0 * DROP_ML     # maximum dose per event (~0.33 ml)
min_ml            =  1.0 * DROP_ML     # minimum dose per event (~0.033 ml)
req_rate_ml_min   = 20.0               # pump flow rate during dosing (ml/min)
cooldown_s        = 5.0                # minimum seconds between doses

# Current pH control parameters configured
print("Parameters configured.")
print(f"  Target pH: {target_pH} ± {epsilon} (re-arms at ±{outer_band})")
print(f"  PID: kp={kp}, ki={ki}, kd={kd}")
print(f"  Dose range: {min_ml:.4f} – {max_ml:.4f} ml at {req_rate_ml_min} ml/min")

## Set up the real-time pH plot

This matplotlib plot is a live plot that updates with each pH reading. This plot shows:

- **Yellow line** — target pH
- **Gray dashed lines** — epsilon (inner band, dispensing disarmed within)
- **Cyan dashed lines** — outer band (re-arm dispensing threshold)

> **Note:** '%matplotlib widget' enables interactive plots. On the left you can select the pan button to move about the graph, the rectangle button to zoom to a selected location, and the home button to reset the plot (removes data points).

In [ ]:
# X-axis will shows 300 seconds (5 minutes) of dispensing events
WINDOW_SEC = 300.0


fig, ax = plt.subplots(figsize=(10, 8))
ax.set_facecolor('black')
fig.patch.set_facecolor('black')


# Graph axis labels and settings
ax.set_xlabel('Time (s)', color='white')
ax.set_ylabel('pH', color='white')
ax.set_title('Real-Time pH Control', color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('white')
ax.set_ylim(0, 14)
ax.set_xlim(0, WINDOW_SEC)
ax.grid(True, alpha=0.3, color='white')

# Guide lines including: target, epsilons, outer bands
line_target      = ax.axhline(target_pH,              color='yellow',    linewidth=1,   linestyle='-')
line_upper_eps   = ax.axhline(target_pH + epsilon,     color='lightgray', linewidth=0.8, linestyle='--')
line_lower_eps   = ax.axhline(target_pH - epsilon,     color='lightgray', linewidth=0.8, linestyle='--')
line_upper_outer = ax.axhline(target_pH + outer_band,  color='cyan',      linewidth=0.8, linestyle='--')
line_lower_outer = ax.axhline(target_pH - outer_band,  color='cyan',      linewidth=0.8, linestyle='--')

# Plot pH data
ph_line, = ax.plot([], [], 'o-', color='lime', linewidth=1, markersize=3)

# Graph legend
legend_elements = [
    mpatches.Patch(color='yellow',    label=f'Target pH ({target_pH})'),
    mpatches.Patch(color='lightgray', label=f'Epsilon (±{epsilon})'),
    mpatches.Patch(color='cyan',      label=f'Outer band (±{outer_band})'),
]
ax.legend(handles=legend_elements, loc='upper right',
          facecolor='black', labelcolor='white', fontsize=8)

plt.tight_layout()
plt.show()
print("Plot ready.")

## Run the real-time pH controller

This is the main pH controller loop. Each iteration:
- Wait 8 seconds - allow for mixing of solution before measuring again, adjust `INTERVAL_S` if needed
- Read pH sensor
- Log reading in output file
- Run PID calculation
- If needed, doses acid or base - subject to hysteresis and a cooldown
- Updates the live plot above

**To stop the loop:** use the Jupyter stop button (■) in the toolbar, or press `I` twice to interrupt the kernel. Both the output file and the graph will be saved if loop is stopped manually or ends at set number of iterations. `N_ITERATIONS` is currenlty set to 300. At 8 seconds per reading, this will last 40 minutes.

In [ ]:
# ---------- Control loop state ----------
err_hist     = deque(maxlen=16)  # stores (time, error) for PID integral/derivative
last_dose_ts = 0.0
can_dose     = True              # hysteresis flag

xs = deque()   # time values for plot
ys = deque()   # pH values for plot

INTERVAL_S   = 8      # seconds between measurements
N_ITERATIONS = 300    # total number of measurements

t0 = time.monotonic()

with open(file_path + file_name, "a") as f:
    print(f"Starting pH control loop ({N_ITERATIONS} readings, {INTERVAL_S}s interval)")
    print(f"Target: pH {target_pH} ± {epsilon} | Output: {file_path + file_name}")
    print("-" * 60)

    try: 
        for i in range(N_ITERATIONS):
    
            # ---- Wait ----
            time.sleep(INTERVAL_S)
    
            # ---- Measure ----
            pH, mV, T = pH_meter.measure_parameters(pH_channel)
            t = time.monotonic() - t0
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log_line = f"[{timestamp}] t={t:.1f}s  pH={pH:.3f}  mV={mV:.3f}  T={T:.2f}°C"
            print(log_line)
            f.write(log_line + "  ")
    
            # ---- PID error ----
            e       = target_pH - pH       # positive = need base, negative = need acid
            abs_err = abs(e)
    
            # ---- Hysteresis flag update ----
            # Disarm when inside epsilon; re-arm only when beyond outer_band
            if abs_err <= epsilon:
                can_dose = False
            elif abs_err >= outer_band:
                can_dose = True
            # else: between epsilon and outer_band — keep previous state
    
            # ---- Error history ----
            err_hist.append((t, e))
    
            # ---- Integral: trapezoid rule over last 10 points ----
            I = 0.0
            pts = list(err_hist)[-10:]
            for j in range(1, len(pts)):
                (t0_p, e0_p), (t1_p, e1_p) = pts[j-1], pts[j]
                dt = max(1e-6, t1_p - t0_p)
                I += 0.5 * (e0_p + e1_p) * dt
    
            # ---- Derivative: average of last two finite differences ----
            D = 0.0
            if len(err_hist) >= 3:
                (t2, e2), (t1, e1), (t0_h, e0_h) = err_hist[-1], err_hist[-2], err_hist[-3]
                d1 = (e2 - e1)   / max(1e-6, t2 - t1)
                d0 = (e1 - e0_h) / max(1e-6, t1 - t0_h)
                D  = 0.5 * (d1 + d0)
            elif len(err_hist) == 2:
                (t1, e1), (t0_h, e0_h) = err_hist[-1], err_hist[-2]
                D = (e1 - e0_h) / max(1e-6, t1 - t0_h)
    
            # ---- PID output ----
            u = kp * e + ki * I + kd * D
    
            # ---- Hysteresis-gated dosing ----
            need_dose = can_dose and (abs_err > epsilon)
            now = time.monotonic()
            if need_dose and (now - last_dose_ts) >= cooldown_s:
                direction = "base" if u > 0 else "acid"
                vol = min(max_ml, max(min_ml, abs(u) * scale_ml_per_unit))
                eff_rate = min(req_rate_ml_min, PUMP_MAX_ML_MIN)
                last_dose_ts = now
    
                if direction == "acid":
                    acid_pump.run(eff_rate, vol, wait=True)
                    dose_msg = f"ACID  {vol:.4f} ml @ {eff_rate:.2f} ml/min"
                else:
                    base_pump.run(eff_rate, vol, wait=True)
                    dose_msg = f"BASE  {vol:.4f} ml @ {eff_rate:.2f} ml/min"
    
                print(f"  → Dose {dose_msg}")
                f.write(f"dispensing {dose_msg}\n")
            else:
                f.write("pH in range\n")
    
            # ---- Update plot ----
            xs.append(t)
            ys.append(pH)
    
            # Trim to window
            while xs and xs[0] < t - WINDOW_SEC:
                xs.popleft()
                ys.popleft()
    
            ph_line.set_data(list(xs), list(ys))
            if t < WINDOW_SEC:
                ax.set_xlim(0, WINDOW_SEC)
            else:
                ax.set_xlim(t - WINDOW_SEC, t)
            fig.canvas.draw_idle()

    except KeyboardInterrupt:
        print("\nLoop stopped manually.")
    finally:
        print("\nControl loop complete.")
        fig.savefig(file_path + "pH_plot.png", dpi=150, bbox_inches='tight', facecolor='black')
        print(f"Graph saved to {file_path}pH_plot.png")

## Empty and clean acid tubing

Repeat the acid pump run for the following steps:
- Remove the input tubing from your acid solution and run your acid pump to empty the tubing
- Place input tubing in clean water and run your acid pump to clean the line with water
- Remove any input liquid and run pump to pull air and empty the line

In [ ]:
# Empty acid tubing
acid_pump.run(6.0, 1.5, wait=True)

In [ ]:
# Prime tubing with water
acid_pump.run(6.0, 1.5, wait=True)

In [ ]:
# Empty water from tubing
acid_pump.run(6.0, 1.5, wait=True)

## Empty and clean base tubing

Repeat the base pump run for the following steps:

- Remove the input tubing from your base solution and run your base pump to empty the tubing
- Place input tubing in clean water and run your base pump to clean the line with water
- Remove any input liquid and run pump to pull air and empty the line

In [ ]:
# Empty base tubing
base_pump.run(6.0, 1.5, wait=True)

In [ ]:
# Prime tubing with water
base_pump.run(6.0, 1.5, wait=True)

In [ ]:
# Empty water from tubing
base_pump.run(6.0, 1.5, wait=True)